# Assignment 2: Milestone I Natural Language Processing
## Task 2&3
#### Student Name: XXXX XXXX
#### Student ID: 000000


Environment: Python 3 and Jupyter notebook

Libraries used: please include all the libraries you used in your assignment, e.g.,:
* pandas
* re
* numpy

## Introduction
You should give a brief information of this assessment task here.

<span style="color: red"> Note that this is a sample notebook only. You will need to fill in the proper markdown and code blocks. You might also want to make necessary changes to the structure to meet your own needs. Note also that any generic comments written in this notebook are to be removed and replace with your own words.</span>

## Importing libraries 

In [3]:
# Code to import libraries as you need in this assessment, e.g.,
import pandas as pd
import os
from collections import Counter
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from gensim.models import KeyedVectors


In [42]:
VOCAB_PATH = "./data/vocab.txt"
DATASET_PATH = "./data/processed.csv"
BOW_PATH = "./data/count_vectors.txt"
UNWEIGHTED_PATH = "./data/unweighted_vectors.txt"
WEIGHTED_PATH = "./data/weighted_vectors.txt"
GLOVE_PATH = "/Users/nhan.ngo/rmit/RMIT-Advanced-Programming-for-Data-Science/archive/glove.6B.300d.txt"
OUTPUT_PATH = "./data/unweighted_vectors.txt"


In [9]:
def load_vocab(path: str) -> dict:
    vocab: dict = {}
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if ":" in line:
                word, idx = line.rsplit(":", 1)
                vocab[word.strip()] = int(idx.strip())
    return vocab  # {"absolutely": 1, "feels": 42, …}

In [4]:
def build_count_vector(review_text: str, vocab: dict[str, int]) -> dict[int, int]:
    tokens = review_text.split() # already cleaned — just split on whitespace
    counts: Counter = Counter()

    for token in tokens:
        if token in vocab:
            counts[vocab[token]] += 1 # accumulate by vocab index

    return dict(counts)

In [5]:
def make_bow_line(row_idx: int, count_vector: dict):
    """ Relied on the requirement from assignment
        ```
        count_vectors.txt This file stores the sparse count vector representation of cosmetics/beauty
            reviews in the following format. Each line of this file corresponds to one review. It starts with
            a ‘#’ key followed by the index of the item review, and a comma ‘,’. The rest of the line is the
            sparse representation of the corresponding description in the form of
            word_integer_index:word_freq separated by comma.
        ```
    """
    if not count_vector:
        return f"#{row_idx},"
    pairs: str = ",".join(f"{idx}:{freq}" for idx, freq in sorted(count_vector.items()))
    return f"#{row_idx},{pairs}"

In [6]:
def make_bows(reviews: str, vocab: dict, path: str):
    content: str = ""
    for i, r in enumerate(reviews):
        r = str(r) # Cast type
        count_vector: dict = build_count_vector(r, vocab)
        line: str = make_bow_line(i, count_vector) + "\n"
        content +=line
    with open(path, "w", encoding="utf-8") as out:
        out.write(content)
    print(f"Saved to {path}")

## Task 2. Generating Feature Representations for Clothing Items Reviews

...... Sections and code blocks on buidling different document feature represetations


<span style="color: red"> You might have complex notebook structure in this section, please feel free to create your own notebook structure. </span>

In [10]:
# Code to perform the task...
vocab: dict = load_vocab(VOCAB_PATH)
df = pd.read_csv(DATASET_PATH)

In [11]:
# Fill na with empty string
df["processed_review_text"] = df["processed_review_text"].fillna('')

### 2.1. Count Logic - Bag of Word
Saving outputs
Save the count vector representation as per spectification.
- count_vectors.txt

In [12]:
make_bows(reviews=df["processed_review_text"].values, vocab=vocab, path=BOW_PATH)

Saved to ./data/count_vectors.txt


### 2.2. Association Logic 
Embedded Model Comparison
| Criterion | GloVe | Word2Vec (GoogleNews-300) | FastText | all-MiniLM-L6-v2 |
|---|---|---|---|---|
| **Model type** | Static word | Static word | Static word | Contextual (Transformer) |
| **Vector size** | 50–300 | 300 | 300 | 384 |
| **Download size** | ~800MB | ~1.6GB | ~8GB | ~80MB |
| **Memory at runtime** | ~500MB | ~3.4GB | ~15GB+ | ~200MB |
| **Out-of-vocab handling** | None — skip word | None — skip word | Subword-based, handles typos | Tokeniser handles all |
| **Context awareness** | No — same vector regardless of context | No — same vector regardless of context | No — same vector regardless of context | Yes — changes per sentence |
| **Trained on** | Wikipedia + Common Crawl | Google News (100B words) | Common Crawl (600B tokens) | 1B+ sentence pairs (diverse) |
| **Review domain fit** | Moderate — general vocab | Weaker — news not reviews | Good — handles informal text | Best — trained on diverse text |
| **Ease of use** | ✅ Easy — manual load, no extra lib | Medium — gensim needed | Medium — gensim/fasttext lib | Easy — 1 line of code |
| **Classification quality** | Decent — baseline level | Decent but domain mismatch | Good — robust to noise | Best — top MTEB scorer |
| **After Task 1 preprocessing** | ✅ Fine — word-level lookup | ⚠️ Acceptable — but domain mismatch worsens | ✅ Best — subwords still help | ❌ Loses key advantage — designed for natural sentences not keyword bags |
| **Paradigm** | Paradigm 2 — Association Logic | Paradigm 2 — Association Logic | Paradigm 2 — Association Logic | Paradigm 3 — Contextual Transformer |
| **Assignment compliant** | ✅ Yes — explicitly listed | ✅ Yes — explicitly listed | ✅ Yes — explicitly listed | ⚠️ Needs justification |
| **Practical feasibility** | ✅ Loads in seconds, ~500MB RAM | ⚠️ Heavy — 3.4GB RAM | ❌ Impractical — 15GB+ RAM exceeds most machines | ✅ Lightweight |
| **Verdict** | ⭐ **Best practical choice** | Weakest choice | Theoretically best but impractical — 15GB+ memory consumption makes it infeasible for this assignment | Not recommended after preprocessing |


**Verdict:** In experiment, we ran the FastText model but it ate lots of memory more than 15GB+, it is still accept in research pharse but in application (web application) it is not suitable. Therefore, we decided to use Glove with balance between performance and cost. 

#### Unweighted document vectors

In [ ]:
wv = KeyedVectors.load_word2vec_format(GLOVE_PATH, binary=False, no_header=True)
dim = wv.vector_size 

In [13]:
def doc_vector_unweighted_glove(wv, text: str, dim: int) -> np.ndarray:
    text = str(text).strip()
    if not text:
        return np.zeros(dim, dtype=np.float32)
    vecs = [wv[t] for t in text.split() if t in wv]
    if not vecs:
        return np.zeros(dim, dtype=np.float32)
    return np.mean(np.vstack(vecs), axis=0).astype(np.float32)

In [18]:
reviews: list[str] = df['processed_review_text'].fillna('').tolist()

In [22]:
def embed_all_unweighted_glove(wv, reviews: list[str], dim: int) -> np.ndarray:
    out = np.zeros((len(reviews), dim), dtype=np.float32)
    for i, r in enumerate(reviews):
        out[i] = doc_vector_unweighted_glove(wv, r, dim)
    return out

In [23]:
def save_vectors_txt(path: str, embeddings: np.ndarray) -> None:
    """One line per row: #idx,v1,v2,..."""
    with open(path, "w", encoding="utf-8") as f:
        for idx, vec in enumerate(embeddings):
            vec_str = ",".join(f"{v:.6f}" for v in vec)
            f.write(f"#{idx},{vec_str}\n")

In [24]:
vectors: np.ndarray = embed_all_unweighted_glove(wv, reviews, dim)

In [26]:
reviews[0]

'works claims difference day olay cleanser results'

In [27]:
vectors[0][:10]

array([ 0.09216715,  0.23001385,  0.01135653,  0.0755953 , -0.25608355,
       -0.02611542, -0.16869442, -0.10300914,  0.02943842, -1.0461543 ],
      dtype=float32)

In [29]:
save_vectors_txt(UNWEIGHTED_PATH, vectors)

#### Weighted (TF‑IDF) document vectors

In [ ]:
def get_tf_idf_vocab(vocab_path: str) -> dict:
    vocab: dict = {}
    with open(vocab_path, 'r') as f:
        for line in f:
            line = line.strip()
            if ':' in line:
                word, idx = line.rsplit(':', 1)  # rsplit to handle words with ':' in them
                vocab[word] = int(idx)
    return vocab


In [33]:
vocab: dict = get_tf_idf_vocab(VOCAB_PATH)
print(list(vocab.items())[:5])

[('aa', 0), ('aback', 1), ('abd', 2), ('abh', 3), ('ability', 4)]


In [ ]:
def tfidf_weighted_glove(reviews: list[str], wv, dim: int, vocab: dict) -> np.ndarray:
    vectorizer = TfidfVectorizer(
        analyzer=lambda s: str(s).split(),
        lowercase=False,
        vocabulary=vocab # fixed to Task 1 vocab
    )
    X = vectorizer.fit_transform(reviews)
    terms = vectorizer.get_feature_names_out()
    n_docs = X.shape[0]
    out = np.zeros((n_docs, dim), dtype=np.float32)
    for i in range(n_docs):
        row = X.getrow(i)
        if row.nnz == 0:
            continue
        acc = np.zeros(dim, dtype=np.float64)
        wsum = 0.0
        for idx, w in zip(row.indices, row.data):
            t = terms[idx]
            if t not in wv:
                continue  # GloVe: skip OOV
            acc += w * wv[t]
            wsum += w
        if wsum > 0:
            out[i] = (acc / wsum).astype(np.float32)
    return out

In [39]:
tfidf_vectors: np.ndarray = tfidf_weighted_glove(reviews, wv, dim, vocab)

In [40]:
reviews[0]

'works claims difference day olay cleanser results'

In [41]:
tfidf_vectors[0][:10]

array([ 0.1093761 ,  0.24135557,  0.03933524,  0.0889417 , -0.29680678,
       -0.04488716, -0.15268289, -0.09343771,  0.06214527, -0.8847524 ],
      dtype=float32)

In [43]:
save_vectors_txt(WEIGHTED_PATH, tfidf_vectors)

## Task 3. Clothing Review Classification

...... Sections and code blocks on buidling classification models based on different document feature represetations. 
Detailed comparsions and evaluations on different models to answer each question as per specification. 

<span style="color: red"> You might have complex notebook structure in this section, please feel free to create your own notebook structure. </span>

In [ ]:
# Code to perform the task...


## Summary
Give a short summary and anything you would like to talk about the assessment tasks here.

## Couple of notes for all code blocks in this notebook
- please provide proper comment on your code
- Please re-start and run all cells to make sure codes are runable and include your output in the submission.   
<span style="color: red"> This markdown block can be removed once the task is completed. </span>